<div dir="rtl">
    
### AI usage log – רשומה 1: בדיקת ההשפעה של קוד הסוכן (agent_code)
- **על מה התייעצנו:** איך לבדוק האם ההבדלים בעלות הממוצעת בין 300 הסוכנים אמיתיים או מקריים.
- **מה ה-AI הציע:** נוסחה שמשווה את הפיזור בין הסוכנים הגדולים לפיזור הצפוי ממקריות, ומחשבת את הרעש לפי הגודל הממוצע של סוכן.
- **הבעיה שמצאנו:** כשהרצנו את הקוד, התוצאה (426 לעומת 340) סתרה את המסקנה שה-AI הציג. התברר שהנוסחה שגויה, כי גודלי הסוכנים שונים מאוד זה מזה, ושימוש בגודל ממוצע מקטין את הרעש הצפוי.
- **איך תיקנו ווידאנו:** החלפנו את הנוסחה בבדיקת ערבוב (permutation test): ערבוב העלויות בין הנהגים 300 פעם ובדיקה אם הפיזור האמיתי חריג. הפיזור האמיתי (807) היה דומה לממוצע הערבובים (803), ו-44% מהערבובים יצאו גדולים ממנו. המסקנה להסיר את העמודה נשארה, אבל עכשיו היא מבוססת על בדיקה נכונה.

### AI usage log – רשומה 2: קריסה של פונקציית הניקוי על עמודה ריקה
- **על מה התייעצנו:** כתיבת חוקי ניקוי לעמודות country ו-area_type בתוך הפונקציה clean_driver_cols, שתשולב ב-prepare_data.
- **מה ה-AI הציע:** שימוש ישיר ב-`.str.strip()` וב-`.str.upper()`, ובדיקת "נהגים מזויפים" עם ערכים שגויים כדי לוודא שהפונקציה לא קורסת.
- **הבעיה שמצאנו:** השותף גילה שכאשר עמודת טקסט ריקה לגמרי, למשל שורה בודדת עם מדינה חסרה, pandas מזהה אותה כעמודה מספרית, ו-`.str` קורס עם AttributeError. ההנחיות דורשות שהפונקציה תרוץ גם על שורה בודדת, ולכן הבאג היה עלול לגרום לאובדן כל נקודות התחרות. הבדיקה שה-AI תכנן לא תפסה את זה, כי שילבה טקסט ו-NaN באותה עמודה.
- **איך תיקנו ווידאנו:** המרה לטקסט לפני פעולות הטקסט, והחזרת הערכים החסרים בסוף (`astype(str)` ואחר כך `.where(notna)`). בסקירה נוספת נמצאו ותוקנו עוד שלוש חולשות: ערך אינסופי שהיה מפיל את ה-Pipeline, מחרוזת ריקה שנשארה כקטגוריה, ומספר עם פסיק. התיקון רוכז בשתי פונקציות עזר, to_num ו-clean_text. וידאנו שהתוצאה על הדאטה זהה, והוספנו למחברת תא בדיקות קיצון: שורה ריקה, קריאה כטקסט, אינסוף, מחרוזות ריקות, 300 שורות בודדות מול הרצה מלאה, אפס שורות ואינדקס כפול. כל הבדיקות עברו.

</div>

<div dir="rtl">

### AI usage log – רשומה 3: אותה קריסה של `.str` גם בעמודות הרכב
- **על מה התייעצנו:** כתיבת חוק ניקוי לעמודה vehicle_type בתוך הפונקציה clean_vehicle_cols.
- **מה ה-AI הציע:** שימוש ישיר ב-`.str.strip().str.lower()`, בדיוק כמו בבאג מרשומה 2.
- **הבעיה שמצאנו:** כשהעברנו לשיחה את המחברת המעודכנת של השותף, ה-AI זיהה שאותה בעיה קיימת גם בחוק שהוא עצמו הציע: עמודה ריקה בשורה בודדת הייתה גורמת לקריסה. הבדיקה הראשונית שלנו (shape על שורה בודדת) לא תפסה את זה, כי בדאטה אין ערכים חסרים בעמודה הזו.
- **איך תיקנו ווידאנו:** קודם הוספנו `astype(str)` ורשימת ערכים חוקיים (`isin`). אחר כך עברנו לפונקציות העזר המשותפות to_num ו-clean_text בכל עמודות הרכב. ה-AI הריץ את בדיקות הקיצון מקובץ הדגשים על הפונקציה, ובנוסף "נהגים מזויפים" עם ערכים כמו " SUV ", "1,200", "₪5000", inf ו-"yes". כל הבדיקות עברו. הלקח: באג שנמצא בחלק אחד של הקוד צריך לחפש בכל החלקים.

### AI usage log – רשומה 4: הצעות של Gemini שנבדקו מול הדאטה ונדחו
- **על מה התייעצנו:** בתחילת העבודה ביקשנו מ-Gemini רשימת עמודות שדורשות תשומת לב, והצעות לטיפול בהן.
- **מה ה-AI הציע:** (1) ב-has_telematics, לאחד את הערך 2 ל-1, בהנחה שזו טעות הקלדה. (2) ב-vehicle_value, ההערכה הייתה שמסתתרים בעמודה תווים כמו ₪ או מילים כמו "unknown".
- **הבעיה שמצאנו:** (1) אם 2 היה בעצם 1, הנהגים האלה היו דומים לקבוצה 1 (גיל ממוצע 44.5). בפועל הגיל הממוצע שלהם (48.0) זהה לממוצע המשוקלל של כל הנהגים, והערך מפוזר באופן אחיד בין המדינות. כלומר, זו תערובת אקראית ולא טעות הקלדה של 1. (2) אחרי הסרת הפסיקים כל 45,000 הערכים הפכו למספרים, כך שהבעיה היחידה הייתה 682 ערכים עם פסיק.
- **איך תיקנו ווידאנו:** (1) הערך 2 הופך לחסר במקום ל-1, וההשלמה נעשית ב-Pipeline. (2) החוק מסיר פסיקים בלבד, ו-`errors="coerce"` מבטיח שכל צורת כתיבה אחרת שתגיע בקובץ הבדיקה תהפוך לחסר ולא תקרוס. הלקח: הצעות של AI הן השערות, וכל אחת נבדקה מול הדאטה לפני שנכנסה לקוד.

</div>

<div dir="rtl">

### AI usage log – רשומה 5: איחוד prepare_data במחברת הראשית
- **על מה התייעצנו:** איך להעביר את פונקציות הניקוי שכתבנו במחברות החקירה למחברת הראשית, ואיך לבדוק שהן לא קורסות על הקובץ של הבודק.
- **מה ה-AI הציע:** לבחור עמודות לפי רשימה של עמודות מותרות במקום למחוק עמודות אסורות, ותא של בדיקות קיצון. המקרים שנבדקים בתא מבוססים על הבאגים שמצאנו קודם (רשומות 2 ו-3).
- **מה שאלנו והחלטנו בעצמנו:** שמנו לב שאין ל-engine_hp תקרה, ושאלנו מה יקרה אם אצל הבודק יופיע ערך קיצוני. בדקנו כמה חלופות (תקרה, NaN, לוג, חיתוך התחזית), והחלטנו לדחות את ההחלטה לבלוק 3, כשנוכל למדוד את ההשפעה ב-CV.
- **איך וידאנו:** הרצנו את הבדיקות בסביבה שלנו (pandas 2.2), וכל 12 עברו. השווינו את הפונקציות במחברת הראשית לאלה שבמחברות החקירה. ה-AI הצביע על כך שיש בינינו פער בגרסת scikit-learn (1.5.1 מול 1.6.1). נחליט על גרסה אחת ונכתוב אותה ב-requirements.txt.

</div>

<div dir="rtl">

### AI usage log – רשומה 6: פיצ'רי הרכב ושילובם ב-prepare_data

- **על מה התייעצנו:** אילו פיצ'רים לבנות מעמודות הרכב, ואיך לבדוק כל אחד מהם.
- **מה ה-AI הציע:** לבדוק כל מועמד בטבלה לפי קבוצות ובתיקוף צולב. אחרי שהלוגים והדגל לא עזרו, הציע לעבור לאינטראקציות, והציע את הרעיון של נהג צעיר כפול עוצמת מנוע.
- **מה שאלנו והחלטנו בעצמנו:** שאלנו מתי לוג עוזר, ולמה דגל שימוש לא משפר כשיש כבר סוג שימוש. דחינו ארבעה מועמדים, וקיבלנו שניים, אחרי שבדקנו שהפער בטבלה גדל באופן עקבי ושיש מספיק נהגים בכל קבוצה (את ק"מ כפול עבודה דחינו כי היו רק 24 ו-61 נהגי עבודה בקבוצות האמצעיות).
- **איך וידאנו:** הרצנו את כל התאים בסביבה שלנו. שני הפיצ'רים שיפרו ב-9 מתוך 10 folds. add_vehicle_features עברה 13 בדיקות קיצון, ואחרי השילוב כל 12 הבדיקות של prepare_data וסימולציית הבודק עברו.

</div>